In [1]:
# import nessory libreris

from pathlib import Path
import argparse
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
"""Train and compare ferry-demand models using chronological evaluation.
Run: python model.py --data "Toronto Island Ferry Tickets.csv"
Optional time-series models: python model.py --data "Toronto Island Ferry Tickets.csv" --include-time-series
Install base packages: pip install pandas numpy scikit-learn
Optional: pip install xgboost statsmodels prophet
"""

from pathlib import Path
import json
import warnings

p = Path("main_02train_test.ipynb")
if not p.exists():
    p = Path("dataProcess/main_02train_test.ipynb")

nb = json.loads(p.read_text(encoding="utf-8"))
ns = {}

for cell in nb["cells"]:
    if cell["cell_type"] == "code":
        code = "".join(cell["source"])
        if code.strip():
            exec(code, ns)

TARGETS = ns["TARGETS"]
HORIZONS = ns["HORIZONS"]
load_data = ns["load_data"]
build_features = ns["build_features"]
make_xy = ns["make_xy"]
time_split = ns["time_split"]
rolling_origin_splits = ns["rolling_origin_splits"]

warnings.filterwarnings("ignore", category=FutureWarning)


In [3]:
def metrics(y_true, y_pred):
    """MAE/RMSE/sMAPE; lower values are better. sMAPE handles zeros safely."""
    a, p = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.abs(a) + np.abs(p)
    smape = np.mean(np.divide(2 * np.abs(a-p), denom, out=np.zeros_like(denom), where=denom != 0)) * 100
    return {"MAE": mean_absolute_error(a, p), "RMSE": np.sqrt(mean_squared_error(a, p)), "sMAPE_percent": smape}




In [4]:
def build_models():
    """Models are seeded where applicable so runs are reproducible."""
    models = {
        "LinearRegression": LinearRegression(),
        "RandomForest": RandomForestRegressor(n_estimators=250, min_samples_leaf=2, max_features=0.8,
                                                random_state=42, n_jobs=-1),
        "GradientBoosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                                        max_depth=3, random_state=42),
    }
    try:
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=6,
                                         subsample=0.8, colsample_bytree=0.8,
                                         objective="reg:squarederror", random_state=42, n_jobs=-1)
    except ImportError:
        print("XGBoost is optional and not installed; skipping it (pip install xgboost).")
    return models



In [5]:
def rolling_validation(estimator, X, y, max_splits=4):
    """Evaluate an estimator on expanding-window folds, not random folds."""
    rows = []
    for fold, (tr, va) in enumerate(rolling_origin_splits(X, max_splits=max_splits), 1):
        model = estimator.__class__(**estimator.get_params())
        model.fit(X.iloc[tr], y.iloc[tr])
        pred = np.maximum(model.predict(X.iloc[va]), 0)  # counts cannot be negative
        rows.append({"fold": fold, **metrics(y.iloc[va], pred)})
    return pd.DataFrame(rows)




In [6]:
def optional_sarima(y_train, n_steps):
    """Optional SARIMA forecast; uses recent history to control runtime."""
    try:
        from statsmodels.tsa.statespace.sarimax import SARIMAX
    except ImportError:
        print("Install statsmodels to use SARIMA: pip install statsmodels"); return None
    try:
        # 96 observations = one day at 15-minute frequency. Limit fit history for runtime.
        series = y_train.astype(float).tail(96 * 60)
        fitted = SARIMAX(series, order=(1, 1, 1), seasonal_order=(1, 0, 1, 96),
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=50)
        return np.asarray(fitted.forecast(steps=n_steps))
    except Exception as e:
        print(f"SARIMA skipped after error: {e}"); return None




In [7]:

def optional_prophet(y_train, forecast_index):
    """Optional Prophet forecast using timestamp-aware daily/weekly/yearly seasonality."""
    try:
        from prophet import Prophet
    except ImportError:
        print("Install Prophet to use it: pip install prophet"); return None
    try:
        # Prophet can be slow with many years of 15-minute rows; use recent history.
        series = y_train.tail(96 * 60)
        frame = pd.DataFrame({"ds": pd.to_datetime(series.index), "y": series.to_numpy(float)})
        m = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True)
        m.fit(frame)
        forecast = m.predict(pd.DataFrame({"ds": pd.to_datetime(forecast_index)}))
        return forecast["yhat"].to_numpy()
    except Exception as e:
        print(f"Prophet skipped after error: {e}"); return None





In [8]:
def main():
    parser = argparse.ArgumentParser()
    default_data_candidates = [
        Path.cwd() / "Toronto_Island_Ferry_Tickets.csv",
        Path.cwd() / "dataProcess" / "Toronto_Island_Ferry_Tickets.csv",
    ]
    default_data = next(
        (candidate for candidate in default_data_candidates if candidate.is_file()),
        default_data_candidates[0],
    )
    parser.add_argument("--data", default=str(default_data))
    parser.add_argument("--output-dir", default="model_outputs")
    parser.add_argument("--test-fraction", type=float, default=0.20)
    parser.add_argument("--rolling-validation", action="store_true", help="Run expanding-window CV (takes longer)")
    parser.add_argument("--include-time-series", action="store_true", help="Try optional SARIMA and Prophet models")
    args, extra_args = parser.parse_known_args()
    if extra_args:
        is_kernel_args = (
            len(extra_args) == 1 and extra_args[0].startswith("--f=")
        ) or (
            len(extra_args) == 2 and extra_args[0] in {"-f", "--f"}
        )
        if not is_kernel_args:
            parser.error(f"unrecognized arguments: {' '.join(extra_args)}")
    out = Path(args.output_dir); out.mkdir(parents=True, exist_ok=True)
    features = build_features(load_data(args.data))
    results, predictions = [], []

    for target in TARGETS:
        for horizon in HORIZONS:
            print(f"\n--- {target}; forecast horizon: {horizon} minutes ---")
            X, y = make_xy(features, target, horizon)
            Xtr, Xte, ytr, yte = time_split(X, y, args.test_fraction)
            print(f"Train: {Xtr.index.min()} -> {Xtr.index.max()} ({len(Xtr):,} rows)")
            print(f"Test:  {Xte.index.min()} -> {Xte.index.max()} ({len(Xte):,} rows)")

            # Naive baseline: last observed training target value.
            baselines = {
                "NaiveLastTrainingValue": np.full(len(yte), float(ytr.iloc[-1])),
                # Fixed training-only moving-average baseline. It is intentionally simple.
                "MovingAverage4TrainingValues": np.full(len(yte), float(ytr.tail(4).mean())),
            }
            for name, pred in baselines.items():
                results.append({"target": target, "horizon_minutes": horizon, "model": name, **metrics(yte, pred)})
                predictions.append(pd.DataFrame({"Timestamp": Xte.index, "target": target,
                    "horizon_minutes": horizon, "model": name, "actual": yte.to_numpy(), "predicted": pred}))

            for name, estimator in build_models().items():
                try:
                    estimator.fit(Xtr, ytr)
                    pred = np.maximum(estimator.predict(Xte), 0)
                    results.append({"target": target, "horizon_minutes": horizon, "model": name, **metrics(yte, pred)})
                    predictions.append(pd.DataFrame({"Timestamp": Xte.index, "target": target,
                        "horizon_minutes": horizon, "model": name, "actual": yte.to_numpy(), "predicted": pred}))
                    m = metrics(yte, pred)
                    print(f"{name}: MAE={m['MAE']:.3f}, RMSE={m['RMSE']:.3f}, sMAPE={m['sMAPE_percent']:.2f}%")
                    if args.rolling_validation:
                        cv = rolling_validation(estimator, Xtr, ytr)
                        cv.to_csv(out / f"rolling_{target.replace(' ', '_')}_{horizon}m_{name}.csv", index=False)
                except Exception as e:
                    print(f"{name} skipped after error: {e}")

            if args.include_time_series:
                # For each horizon, y already represents the value at that future horizon.
                # These are comparative univariate baselines; treat results cautiously because
                # horizon-shifted targets are not the same as direct multi-step deployment.
                for name, pred in [("SARIMA", optional_sarima(ytr, len(yte))),
                                   ("Prophet", optional_prophet(ytr, Xte.index))]:
                    if pred is not None and len(pred) == len(yte):
                        pred = np.maximum(np.asarray(pred, float), 0)
                        results.append({"target": target, "horizon_minutes": horizon, "model": name, **metrics(yte, pred)})
                        predictions.append(pd.DataFrame({"Timestamp": Xte.index, "target": target,
                            "horizon_minutes": horizon, "model": name, "actual": yte.to_numpy(), "predicted": pred}))

    result_df = pd.DataFrame(results).sort_values(["target", "horizon_minutes", "MAE"])
    result_df.to_csv(out / "model_comparison.csv", index=False)
    if predictions:
        pd.concat(predictions, ignore_index=True).to_csv(out / "test_predictions.csv", index=False)
    if not result_df.empty:
        best = result_df.loc[result_df.groupby(["target", "horizon_minutes"])["MAE"].idxmin()]
        print("\nBest holdout model per target/horizon (lowest MAE):")
        print(best[["target", "horizon_minutes", "model", "MAE", "RMSE", "sMAPE_percent"]].to_string(index=False))
    print(f"\nResults saved in: {out.resolve()}")
    print("Accuracy is measured on later timestamps; no model can be guaranteed accurate before evaluation.")

if __name__ == "__main__": main()





--- Sales Count; forecast horizon: 15 minutes ---
Train: 2016-07-24 03:15:00 -> 2025-06-07 11:30:00 (5,168 rows)
Test:  2025-06-07 11:45:00 -> 2025-08-31 03:30:00 (1,293 rows)
LinearRegression: MAE=34.625, RMSE=52.061, sMAPE=41.92%
RandomForest: MAE=31.805, RMSE=51.124, sMAPE=31.69%
GradientBoosting: MAE=34.463, RMSE=55.995, sMAPE=34.91%
XGBoost: MAE=32.071, RMSE=51.442, sMAPE=31.66%

--- Sales Count; forecast horizon: 30 minutes ---
Train: 2016-07-16 05:15:00 -> 2025-06-07 11:45:00 (5,153 rows)
Test:  2025-06-07 12:00:00 -> 2025-08-31 03:15:00 (1,289 rows)
LinearRegression: MAE=36.839, RMSE=54.830, sMAPE=43.71%
RandomForest: MAE=32.927, RMSE=51.128, sMAPE=32.77%
GradientBoosting: MAE=33.958, RMSE=52.847, sMAPE=34.75%
XGBoost: MAE=32.402, RMSE=50.939, sMAPE=32.71%

--- Sales Count; forecast horizon: 60 minutes ---
Train: 2016-07-16 05:15:00 -> 2025-06-07 11:45:00 (5,128 rows)
Test:  2025-06-07 12:00:00 -> 2025-08-31 04:15:00 (1,283 rows)
LinearRegression: MAE=41.694, RMSE=60.469, sMAP